# Lab 12: Monitor and Trace Your GenAI Agent

> **Source:** Microsoft Learning -- [mslearn-genaiops](https://github.com/MicrosoftLearning/mslearn-genaiops), `docs/05-monitoring-tracing.md`
> **License:** MIT

## Objectives

By the end of this lab you will:

1. Instrument your application with **OpenTelemetry**
2. Configure the **Azure Monitor exporter** for trace ingestion
3. Analyze **trace trees** showing parent-child span relationships
4. Compare agent versions using **KQL queries** in Log Analytics

| Estimated Time | Estimated Cost |
|---|---|
| ~40 minutes | ~$1--2 |

## Architecture

```mermaid
graph LR
    APP[Python App] -->|OpenTelemetry SDK| EXP[Azure Monitor<br/>Exporter]
    EXP --> AI[Application Insights]
    AI --> LA[Log Analytics]
    LA --> KQL[KQL Queries]
    OI[OpenAIInstrumentor] -->|auto-captures| EXP
```

The telemetry stack has three layers:

| Layer | Tool | Role |
|---|---|---|
| **Instrumentation** | OpenTelemetry SDK + OpenAIInstrumentor | Creates spans with timing, token counts, and metadata |
| **Export** | Azure Monitor Exporter | Ships spans to Application Insights |
| **Analysis** | Log Analytics + KQL | Query, aggregate, and visualize traces |

OpenTelemetry is **vendor-neutral** -- the same instrumentation code works with Jaeger, Zipkin, Datadog, or Azure Monitor by swapping the exporter.

## Step 1: Configure OpenTelemetry

Two lines of code instrument your entire application:

```python
configure_azure_monitor(connection_string=os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING"))
OpenAIInstrumentor().instrument()
```

- **`configure_azure_monitor()`** -- sets up the OpenTelemetry SDK with the Azure Monitor exporter. One-liner that handles tracer providers, span processors, and exporters.
- **`OpenAIInstrumentor().instrument()`** -- monkey-patches the OpenAI SDK to **auto-capture** every API call as a span, including model name, token counts, and latency.

> **Exam Tip:** OpenTelemetry is **vendor-neutral** -- it is an open standard, not an Azure-specific technology. The Azure Monitor exporter is just one of many possible backends. The exam may ask about OpenTelemetry's relationship to vendor lock-in.

In [ ]:
import os
from dotenv import load_dotenv
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry.instrumentation.openai import OpenAIInstrumentor

load_dotenv()
configure_azure_monitor(connection_string=os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING"))
OpenAIInstrumentor().instrument()
print("OpenTelemetry configured with Azure Monitor exporter")
print("OpenAI SDK auto-instrumented")

## Step 2: Run Monitoring Script

The full monitoring script runs **4 versions x 5 test prompts = 20 interactions**, each producing a trace:

```bash
python scripts/run_monitoring.py
```

The span hierarchy looks like this:

```
trail-guide-v1                        # top-level span (one per version)
  |- test-day-hike-gear               # child span (one per test)
  |    |- openai.chat.completions     # auto-instrumented OpenAI call
  |- test-overnight-camping
  |    |- openai.chat.completions
  ...
trail-guide-v2
  |- test-day-hike-gear
  ...
```

Each span captures:
- **Duration** (milliseconds)
- **Custom attributes** (`agent.version`, `test.id`, `response.total_tokens`)
- **Nested auto-instrumented spans** from OpenAIInstrumentor (model, token breakdown)

In [ ]:
print("Run the full monitoring script with: python scripts/run_monitoring.py")
print("This generates 4 top-level spans + 20 child spans + nested OpenAI spans")
print()
print("After traces are ingested (2-5 min), view the tree with:")
print("  python scripts/check_traces.py")

## Step 3: View Trace Trees

Run `check_traces.py` to fetch traces from Application Insights and render the parent-child tree in the terminal:

```bash
python scripts/check_traces.py
```

This queries Log Analytics using the Azure Monitor Query SDK and displays the span hierarchy with durations and token counts.

## Step 4: Analyze in Azure Monitor

Open **Log Analytics** in the Azure portal and run KQL queries to compare versions:

```kql
dependencies
| where name contains "trail-guide"
| summarize avg(duration), avg(customDimensions.total_tokens) by customDimensions.version
```

Expected results for V3 vs V4:

| Version | Avg Tokens | Avg Latency | Notes |
|---|---|---|---|
| V3 | ~850 | ~2.1s | Verbose, 4-section output |
| V4 | ~490 | ~1.4s | Concise, 42% token reduction |

The **42% token reduction** from Lab 10's prompt optimization is now **visible in production telemetry** -- fewer tokens means lower latency and lower cost.

### Evaluation vs Monitoring

| Aspect | Evaluation (Lab 11) | Monitoring (Lab 12) |
|---|---|---|
| **Purpose** | Quality measurement | Operational observability |
| **When** | Pre-deployment (CI/CD) | Post-deployment (production) |
| **Metrics** | Intent, Relevance, Groundedness | Latency, tokens, errors, throughput |
| **Data** | Curated test datasets | Live production traffic |

## Key Takeaways

| Concept | Detail |
|---|---|
| **OpenTelemetry** | Vendor-neutral observability standard. Same instrumentation, swap exporters for different backends. |
| **OpenAIInstrumentor** | Auto-captures every OpenAI SDK call as a span -- model, tokens, latency -- with zero code changes. |
| **Span Hierarchies** | Parent-child relationships (version > test > OpenAI call) show the full execution path of each request. |
| **Evaluation vs Monitoring** | Evaluation = quality (pre-deploy). Monitoring = operations (post-deploy). Both are required. |
| **Ingestion Delay** | Application Insights has a **2--5 minute ingestion delay**. Traces are not instantly queryable. |

---

**Next:** [Lab 13 -- Fine-Tuning Strategies](../lab13-fine-tuning/lab13-fine-tuning.ipynb) -- when and how to go beyond prompt engineering.